<u><b><h1 style="text-align:center; line-height:25px; color:#000000; background:#EFEFEF; border: 1px solid #FF6B6B ; padding:20px;">Sentiment analysis of customer reviews</h1></b></u>
**Course:** (DLBDSEAIS – Project: Artificial Intelligence  
**Tools**: Pandas, scikit-learn for the TF-IDF baseline, Hugging Face transformers (with PyTorch) for DistilBERT and Gradio for the interface  
**Dataset:** <a href="https://www.kaggle.com/datasets/fawadhossaini1415/amazon-fashion-800k-user-reviews-dataset">Amazon Fashion Products Reviews</a>  
**<a href="https://github.com/davidlupau/sentiment-analysis-customer-reviews">GitHub repository</a>**

<b><h2 style="padding: 10px; border-left: 3px solid #FF6B6B;">Setup & Imports</h2></b>

In [ ]:
# Core
import re
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

print("Import successful")

### Select device
Select the compute backend automatically so the notebook runs unchanged on a cloud GPU (CUDA), an Apple Silicon Mac (MPS), or any CPU-only machine.

In [ ]:
def detect_device() -> torch.device:
    """Detect the best available compute device.

    Checks hardware availability in order of preference:
      1. CUDA  – NVIDIA GPU (fastest for most deep-learning workloads).
      2. MPS   – Apple Silicon GPU (Metal Performance Shaders).
      3. CPU   – universal fallback.

    Returns:
        torch.device: The most capable device available on this machine.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = detect_device()
SAMPLES_PER_CLASS = 5_000

print(f"Device            : {DEVICE}")
print(f"Platform          : {platform.platform()}")
print(f"PyTorch           : {torch.__version__}")
print(f"Samples per class : {SAMPLES_PER_CLASS:,}")

### Loading the dataset

In [ ]:
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").is_dir()
)

df = pd.read_csv(PROJECT_ROOT / "data" / "amazon-fashion-reviews-dataset.csv")
print("Dataset loaded successfully")

### Data cleaning

In [22]:
def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Remove unusable rows from the dataset.

    Drops rows where 'text' or 'rating' contain null values, then removes
    exact duplicate rows. Reports counts at each step.

    Args:
        df: Raw DataFrame loaded from the CSV file.

    Returns:
        pd.DataFrame: Cleaned DataFrame with nulls and duplicates removed.
    """
    initial_rows = len(df)
    print(f"Initial row count: {initial_rows:,}")

    null_mask = df[["text", "rating"]].isnull().any(axis=1)
    null_count = int(null_mask.sum())
    if null_count:
        print(f"Dropping {null_count:,} rows with null values in 'text' or 'rating'.")
        df = df[~null_mask]
    else:
        print("No null values found in 'text' or 'rating'.")

    duplicate_count = int(df.duplicated().sum())
    if duplicate_count:
        print(f"Dropping {duplicate_count:,} duplicate rows.")
        df = df.drop_duplicates()
    else:
        print("No duplicate rows found.")

    cleaned_rows = len(df)
    print(f"Final row count: {cleaned_rows:,} ({initial_rows - cleaned_rows:,} rows removed).\n")

    return df.reset_index(drop=True)


df = clean_dataset(df)

Initial row count: 867,310
Dropping 298 rows with null values in 'text' or 'rating'.
Dropping 5,264 duplicate rows.
Final row count: 861,748 (5,562 rows removed).



### Rating to sentiment mapping

In [23]:
def map_rating_to_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    """Add a 'sentiment' column derived from the 'rating' column.

    Mapping:
        1, 2  → negative
        3     → neutral
        4, 5  → positive
    """
    mapping = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}
    df = df.copy()
    df["sentiment"] = df["rating"].map(mapping)
    counts = df["sentiment"].value_counts()
    print("Sentiment distribution:")
    for label in ["positive", "neutral", "negative"]:
        print(f"  {label}: {counts.get(label, 0):,}")
    print()
    return df


df = map_rating_to_sentiment(df)
df.head()

Sentiment distribution:
  positive: 345,569
  neutral: 172,247
  negative: 343,932



,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchases,target,sentiment
0,1.0,It say 5 pair when purchasing but only get 2 r...,I was looking for 5 pair and only received 2 p...,[],B07QFTMTLP,B07QFTMTLP,AHASEZ65RESN57BMGRV6QBM5DTIA,1565088068852,0,True,-1,negative
1,1.0,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after...,[],B0764KKDN1,B0764KKDN1,AE3AMA3QSOHFKV46JJAHTHMMIR6A,1622416429592,0,True,-1,negative
2,1.0,Small,Retuned is too small for me,[],B07J1WHVCP,B07J1WHVCP,AH4CFWQE2HTC5BSWIEF3LVLUFK6A,1565284666220,0,True,-1,negative
3,1.0,Pre-Used When Received,This product came with the sleeves turned insi...,[],B0773JWP64,B0773JWP64,AFEKQFJWST6MVTKEJBQKUUBTWK7A,1581963636172,0,False,-1,negative
4,1.0,Worn once and several places at seams have com...,Worn once and several places at seams have com...,[],B099NST9RX,B08JGNS1NK,AGU2FPKN6ARXUSSGBT6WTVLZKJSQ,1640895438476,0,True,-1,negative


### Class imbalance

In [ ]:
def plot_class_imbalance(df):
    counts = df["sentiment"].value_counts().reindex(["positive", "neutral", "negative"])
    total = counts.sum()

    fig, ax = plt.subplots(figsize=(7, 5))
    colors = ["#4CAF50", "#9E9E9E", "#F44336"]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=0.8)

    for bar, count in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + total * 0.004,
            f"{count:,}\n({count / total * 100:.1f}%)",
            ha="center", va="bottom", fontsize=10,
        )

    ax.set_title("Sentiment Class Distribution", fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Sentiment", fontsize=11)
    ax.set_ylabel("Number of reviews", fontsize=11)
    ax.set_ylim(0, counts.max() * 1.18)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()

    output_dir = PROJECT_ROOT / "analysis_output"
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / "class_imbalance.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"Plot saved to {output_path}")
    plt.show()


plot_class_imbalance(df)

In [ ]:
_ILLEGAL_CHARS_RE = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]')

output_dir = PROJECT_ROOT / "analysis_output"
output_dir.mkdir(exist_ok=True)

df_excel = df.copy()
for col in df_excel.select_dtypes(include='object').columns:
    df_excel[col] = df_excel[col].str.replace(_ILLEGAL_CHARS_RE, '', regex=True)

output_file = output_dir / "sentiment_mapped.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_excel.to_excel(writer, sheet_name="Data", index=False)

print(f"Saved to {output_file}")

### Train / test split

In [ ]:
def split_dataset(
    df: pd.DataFrame,
    test_size: float = 0.2,
    random_state: int = 42,
) -> tuple:
    print(f"Splitting dataset — test size: {test_size:.0%}, random_state: {random_state}")
    df_train, df_test = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df["sentiment"]
    )
    df_train = df_train.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    print(f"  Train : {len(df_train):,} rows")
    print(f"  Test  : {len(df_test):,} rows")
    print("Test set distribution (realistic, kept imbalanced):")
    test_counts = df_test["sentiment"].value_counts()
    total_test = len(df_test)
    for cls in ["positive", "neutral", "negative"]:
        n = test_counts.get(cls, 0)
        print(f"  {cls}: {n:,} ({n / total_test * 100:.1f}%)")
    print()
    return df_train, df_test


df_train, df_test = split_dataset(df)

### Training set balancing

In [ ]:
def balance_training_set(
    df_train: pd.DataFrame,
    n_per_class: int,
    random_state: int = 42,
) -> pd.DataFrame:
    classes = ["positive", "neutral", "negative"]

    print(f"Balancing training set — target: {n_per_class:,} rows per class.")
    print("Before:")
    counts_before = df_train["sentiment"].value_counts()
    for cls in classes:
        print(f"  {cls}: {counts_before.get(cls, 0):,}")

    samples = []
    for cls in classes:
        cls_df = df_train[df_train["sentiment"] == cls]
        available = len(cls_df)
        n = min(n_per_class, available)
        if available < n_per_class:
            print(f"Warning: '{cls}' has only {available:,} rows — using all available.")
        samples.append(cls_df.sample(n=n, random_state=random_state))

    df_balanced = (
        pd.concat(samples)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    print("\nAfter:")
    counts_after = df_balanced["sentiment"].value_counts()
    for cls in classes:
        print(f"  {cls}: {counts_after.get(cls, 0):,}")
    print(f"Total training rows: {len(df_balanced):,}\n")

    return df_balanced


df_train = balance_training_set(df_train, SAMPLES_PER_CLASS)

---